# Run the full Phase 1-3 VOID pipeline in Colab

This notebook launches the current FastAPI/Celery UI and can also run the full command-line pipeline directly in Colab.

It is updated for:

- Phase 1: VOID timing validation and optional quality restoration.
- Phase 2: crowded-human BoT-SORT/ReID tracking with YOLO person detection.
- Phase 3: Ollama VLM-assisted affected-region quadmasks with heuristic fallback and mode-aware prompts.

The current VLM path uses local Ollama on Colab L4 by default. Run the Ollama server cell first, then the model-download cell, then the preflight/app cells. If Ollama is unavailable at runtime, the pipeline falls back to deterministic heuristic affected-region masks.


## 1. Mount Drive And Configure Paths

Set `PROJECT_ROOT_DIR` to the folder containing this repository's `main.py`. If the repo is already copied to `/content`, leave the candidates in place and the setup cell will auto-detect it.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import atexit
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import urllib.request

# Change this if your Drive folder is different.
PROJECT_ROOT_DIR = "/content/drive/MyDrive/LiveInterview"

PROJECT_ROOT_CANDIDATES = [
    Path(PROJECT_ROOT_DIR),
    Path("/content/drive/MyDrive/VideoObjectRemoval"),
    Path("/content/VideoObjectRemoval"),
    Path.cwd(),
]

PROJECT_ROOT = None
for candidate in PROJECT_ROOT_CANDIDATES:
    if candidate and (candidate / "main.py").exists():
        PROJECT_ROOT = candidate.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root with main.py. Update PROJECT_ROOT_DIR above."
    )

PROJECT_ROOT_DIR = str(PROJECT_ROOT)
os.environ["PROJECT_ROOT"] = PROJECT_ROOT_DIR
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

VOID_REPO = Path("/content/void-model")
SAM2_REPO = Path("/content/sam2")
SAMURAI_REPO = Path("/content/samurai")
COLAB_OUTPUT_ROOT = Path("/content/video_object_removal_outputs")
COLAB_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
COLAB_CONSTRAINTS = Path("/content/video_object_removal_colab_constraints.txt")
REID_WEIGHTS_DIR = PROJECT_ROOT / "models" / "reid"
REID_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

RESOURCE_PROFILE = "l4_pro_balanced"
QUALITY_RESTORATION = "realesrgan"  # none | ffmpeg_bicubic | realesrgan | venhancer
INSTALL_REALESRGAN = True
REALESRGAN_REPO = Path("/content/Real-ESRGAN")
REALESRGAN_WRAPPER = Path("/content/run_realesrgan_restore.py")
REALESRGAN_MODEL = "RealESRGAN_x4plus"
REALESRGAN_OUTSCALE = "1"
REALESRGAN_TILE = "128"
REALESRGAN_WORKERS = "6"  # Adaptive retry starts here, then falls back to 4, 2, and 1 if needed.
VENHANCER_REPO = Path("/content/VEnhancer")
VENHANCER_WRAPPER = Path("/content/run_venhancer_restore.py")
INSTALL_VENHANCER = False  # Set True only after confirming the selected VEnhancer repo/checkpoints for your Colab runtime.
VENHANCER_GIT_URL = "https://github.com/Vchitect/VEnhancer.git"
VENHANCER_WORKERS = "4"  # Adaptive frame retry starts here, then falls back to 2 and 1.
VENHANCER_INFERENCE_COMMAND_TEMPLATE = ""  # Whole-video fallback command that writes {raw_output}.
VENHANCER_FRAME_COMMAND_TEMPLATE = ""  # Per-frame command that reads {input} and writes {output}. Enables parallel workers.
VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE = ""  # Optional batch command that reads {input_dir} and writes frames to {output_dir}.
VENHANCER_COMMAND_TEMPLATE = ""  # Set this when QUALITY_RESTORATION = "venhancer".
VLM_PROVIDER = "ollama"
OLLAMA_HOST = "127.0.0.1:11434"
OLLAMA_BASE_URL = f"http://{OLLAMA_HOST}"
OLLAMA_VLM_MODEL = "llama3.2-vision:11b"
OLLAMA_VLM_FALLBACK_MODEL = "llava:7b"
OLLAMA_MODELS_DIR = Path("/content/ollama_models")
OLLAMA_VLM_SAMPLE_FRAMES = "3"
OLLAMA_VLM_MAX_IMAGE_SIDE = "768"
OLLAMA_VLM_TIMEOUT_SECONDS = "180"
OLLAMA_VLM_KEEP_ALIVE = "0"

# L4-friendly crowded-human defaults from the Phase 2/3 plan.
CROWDED_HUMAN_DETECTOR_MODEL = "yolo11m.pt"
CROWDED_HUMAN_REID_MODEL = "osnet_x0_25_msmt17.pt"
CROWDED_HUMAN_REID_URLS = {
    "osnet_x0_25_msmt17.pt": "https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF",
}
CROWDED_HUMAN_DETECTOR_IMGSZ = "960"
CROWDED_HUMAN_PERSON_CONF = "0.45"
CROWDED_HUMAN_PERSON_IOU = "0.70"
CROWDED_HUMAN_TARGET_ROI_MIN_IOU = "0.25"
CROWDED_HUMAN_SAM2_IMAGE_SIZE = "1024"
CROWDED_HUMAN_SAM2_REFINE_EVERY_N_FRAMES = "1"
CROWDED_HUMAN_MISSING_TARGET_WARNING_FRAMES = "3"
CROWDED_HUMAN_NEIGHBOR_EXCLUSION = "true"
CROWDED_HUMAN_NEIGHBOR_MASK_EROSION_PX = "3"

print(json.dumps({
    "project_root": PROJECT_ROOT_DIR,
    "void_repo": str(VOID_REPO),
    "sam2_repo": str(SAM2_REPO),
    "samurai_repo": str(SAMURAI_REPO),
    "resource_profile": RESOURCE_PROFILE,
    "quality_restoration": QUALITY_RESTORATION,
    "realesrgan_repo": str(REALESRGAN_REPO),
    "realesrgan_model": REALESRGAN_MODEL,
    "realesrgan_workers": REALESRGAN_WORKERS,
    "venhancer_repo": str(VENHANCER_REPO),
    "venhancer_workers": VENHANCER_WORKERS,
    "vlm_provider": VLM_PROVIDER,
    "ollama_model": OLLAMA_VLM_MODEL,
    "ollama_base_url": OLLAMA_BASE_URL,
    "crowded_human_detector": CROWDED_HUMAN_DETECTOR_MODEL,
    "crowded_human_reid": CROWDED_HUMAN_REID_MODEL,
    "reid_weights_dir": str(REID_WEIGHTS_DIR),
}, indent=2))


## 2. Helpers And GPU Check


In [ ]:
def run(cmd, cwd=None, env=None, check=True):
    cmd = [str(item) for item in cmd]
    print("$", " ".join(cmd))
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=check,
    )


def run_stream(cmd, cwd=None, env=None, check=True):
    cmd = [str(item) for item in cmd]
    print("$", " ".join(cmd))
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line.rstrip())
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}: {' '.join(cmd)}")
    return return_code


try:
    run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found. In Colab, switch Runtime -> Change runtime type -> GPU.")

print("Python:", sys.executable)
print("CWD:", Path.cwd())


## 3. Install System And Python Dependencies

This installs the repo requirements plus the optional Phase 2 tracker dependencies: `ultralytics` and `boxmot`.


In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "redis-server", "git-lfs"])
run(["service", "redis-server", "start"])

# Colab Python 3.12 is sensitive to mixed NumPy/OpenCV wheels. Keep one ABI
# family across project, Ultralytics, BoxMOT, SAM2, and VOID installs.
COLAB_CONSTRAINTS.write_text("""numpy==1.26.4
opencv-python==4.10.0.84
opencv-python-headless==4.10.0.84
""")
print("Wrote Colab constraints:", COLAB_CONSTRAINTS)
print(COLAB_CONSTRAINTS.read_text())


def pip_install(args, check=True):
    # Use streaming output for pip so resolver/build errors are visible in Colab.
    cmd = [sys.executable, "-m", "pip", "install", *args]
    return run(cmd, check=check)


pip_upgrade_result = pip_install(["-U", "pip"], check=False)
if pip_upgrade_result.returncode != 0:
    print(
        "pip self-upgrade failed in this Colab runtime; continuing with the bundled pip. "
        "This is usually okay because the notebook pins the actual package ABI below."
    )
    run([sys.executable, "-m", "pip", "--version"], check=False)
pip_install([
    "--no-cache-dir",
    "--force-reinstall",
    "-c",
    str(COLAB_CONSTRAINTS),
    "numpy==1.26.4",
    "opencv-python==4.10.0.84",
    "opencv-python-headless==4.10.0.84",
])
pip_install([
    "-c",
    str(COLAB_CONSTRAINTS),
    "-r",
    str(PROJECT_ROOT / "requirements.txt"),
])

# Keep BoxMOT on v19 for this Colab stack. BoxMOT v20 requires NumPy >=2.2.0,
# while this notebook pins NumPy 1.26.4 to avoid Colab Python 3.12 ABI crashes.
# Do not downgrade to boxmot==10.x because it pins numpy==1.24.4, which has no
# cp312 wheel and fails to build in current Colab runtimes. Install Ultralytics
# separately so a BoxMOT resolver issue does not hide the YOLO install logs.
pip_install(["-U", "-c", str(COLAB_CONSTRAINTS), "ultralytics", "pyngrok", "gdown"])
run([sys.executable, "-m", "pip", "uninstall", "-y", "boxmot"], check=False)

boxmot_result = pip_install(["-U", "-c", str(COLAB_CONSTRAINTS), "boxmot>=19,<20"], check=False)
if boxmot_result.returncode != 0:
    print("BoxMOT v19 dependency resolution failed with the NumPy constraint.")
    print("Installing BoxMOT runtime dependencies under the same ABI, then BoxMOT with --no-deps.")
    pip_install([
        "-U",
        "-c",
        str(COLAB_CONSTRAINTS),
        "filterpy>=1.4.5,<2",
        "ftfy>=6.1.3,<7",
        "gdown>=5.1.0,<6",
        "gitpython>=3.1.42,<4",
        "huggingface-hub",
        "lapx>=0.5.5",
        "loguru>=0.7,<1",
        "pandas",
        "pyyaml",
        "regex",
        "rich>=13.0.0",
        "scikit-learn>=1.3.0,<2",
        "click>=8.1.8",
        "yacs",
    ])
    pip_install(["-U", "--no-deps", "boxmot>=19,<20"])

# Re-apply the NumPy/OpenCV pins after optional packages, in case a dependency
# resolver attempted to move them. If this cell was run after a broken install,
# restart the runtime once after it completes and rerun from the top.
pip_install([
    "--no-cache-dir",
    "--force-reinstall",
    "numpy==1.26.4",
    "opencv-python==4.10.0.84",
    "opencv-python-headless==4.10.0.84",
])

# Validate in a fresh interpreter, so we catch the exact ABI/package error before YOLO.
run([
    sys.executable,
    "-c",
    (
        "import importlib.metadata as md; "
        "import numpy as np; "
        "print('numpy', np.__version__, np.__file__); "
        "np.random.RandomState(0).rand(1); "
        "import cv2; print('cv2', cv2.__version__); "
        "import ultralytics, boxmot; "
        "print('ultralytics', ultralytics.__version__); "
        "print('boxmot_version', md.version('boxmot')); "
        "print('boxmot', getattr(boxmot, '__file__', '<unknown>'))"
    ),
])

if "numpy" in sys.modules:
    print("NOTE: NumPy was already imported in this kernel. After this repair cell finishes, use Runtime -> Restart runtime, then rerun from the top.")
print("Base project, API, YOLO, and BoT-SORT/ReID dependencies installed.")


## 4. Install Ollama

Run this before downloading the VLM. It installs the `ollama` CLI and points Ollama at a Colab-local model cache.


In [ ]:
def configure_ollama_environment():
    """Apply Ollama settings used by both the notebook and the FastAPI/Celery app."""
    env = os.environ.copy()
    if VLM_PROVIDER != "ollama":
        return env

    OLLAMA_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    values = {
        "OLLAMA_HOST": OLLAMA_HOST,
        "OLLAMA_BASE_URL": OLLAMA_BASE_URL,
        "OLLAMA_VLM_MODEL": OLLAMA_VLM_MODEL,
        "OLLAMA_MODELS": str(OLLAMA_MODELS_DIR),
        "OLLAMA_KEEP_ALIVE": "5m",
        "OLLAMA_NUM_PARALLEL": "1",
        "OLLAMA_VLM_SAMPLE_FRAMES": OLLAMA_VLM_SAMPLE_FRAMES,
        "OLLAMA_VLM_MAX_IMAGE_SIDE": OLLAMA_VLM_MAX_IMAGE_SIDE,
        "OLLAMA_VLM_TIMEOUT_SECONDS": OLLAMA_VLM_TIMEOUT_SECONDS,
        "OLLAMA_VLM_KEEP_ALIVE": OLLAMA_VLM_KEEP_ALIVE,
    }
    os.environ.update(values)
    env.update(values)
    return env


def add_ollama_bin_to_path(bin_dir: Path):
    bin_dir = bin_dir.resolve()
    current_parts = os.environ.get("PATH", "").split(os.pathsep)
    if str(bin_dir) not in current_parts:
        os.environ["PATH"] = str(bin_dir) + os.pathsep + os.environ.get("PATH", "")
    return shutil.which("ollama")


def install_ollama_for_colab():
    existing = shutil.which("ollama")
    if existing:
        print("Ollama binary already installed:", existing)
        return existing

    print("Installing zstd (required by the latest Ollama installer)...")
    run(["apt-get", "install", "-y", "-qq", "zstd"], check=False)

    installer_path = Path("/content/ollama_install.sh")
    print("Downloading Ollama installer...")
    run(["curl", "-fsSL", "https://ollama.com/install.sh", "-o", str(installer_path)])

    print("Running Ollama installer...")
    installer_result = run(["sh", str(installer_path)], check=False)
    installed = shutil.which("ollama")

    if installed:
        if installer_result.returncode != 0:
            print(f"Ollama installer exited with {installer_result.returncode}, but the binary was installed successfully.")
        return installed

    raise RuntimeError(f"Ollama installer failed with exit code {installer_result.returncode}. No binary found.")


if VLM_PROVIDER == "ollama":
    configure_ollama_environment()
    ollama_binary = install_ollama_for_colab()
    version_result = run([ollama_binary, "--version"], check=False)
    if version_result.returncode != 0:
        print("Ollama version check returned nonzero, but continuing because the binary exists:", ollama_binary)
    print("Ollama binary:", shutil.which("ollama") or ollama_binary)
else:
    print("Ollama install skipped because VLM_PROVIDER is not 'ollama'.")

## 5. Start Ollama Server

Start `ollama serve` first and leave it running. The next cell downloads the VLM into this server.


In [ ]:
def wait_for_ollama(timeout=120):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=2) as response:
                if response.status == 200:
                    return True
        except Exception as exc:
            last_error = exc
            time.sleep(1)
    raise RuntimeError(f"Ollama did not become ready at {OLLAMA_BASE_URL}: {last_error}")


if VLM_PROVIDER == "ollama":
    ollama_env = configure_ollama_environment()
    if shutil.which("ollama") is None:
        raise RuntimeError("Ollama is not installed. Run the previous cell first.")

    if "OLLAMA_SERVER_PROCESS" in globals() and OLLAMA_SERVER_PROCESS.poll() is None:
        print("Ollama server already running.")
    else:
        if "OLLAMA_SERVER_LOG_FILE" in globals():
            try:
                OLLAMA_SERVER_LOG_FILE.close()
            except Exception:
                pass
        OLLAMA_SERVER_LOG_PATH = Path("/content/ollama_server.log")
        OLLAMA_SERVER_LOG_FILE = open(OLLAMA_SERVER_LOG_PATH, "a", buffering=1)
        OLLAMA_SERVER_PROCESS = subprocess.Popen(
            ["ollama", "serve"],
            env=ollama_env,
            stdout=OLLAMA_SERVER_LOG_FILE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        print("Ollama server starting. Logs:", OLLAMA_SERVER_LOG_PATH)

    wait_for_ollama()
    print("Ollama server ready:", OLLAMA_BASE_URL)
else:
    print("Ollama server skipped because VLM_PROVIDER is not 'ollama'.")


## 6. Download The Ollama VLM

Pull the selected vision model after the server is ready. The pipeline uses the final `OLLAMA_VLM_MODEL` value from this cell.


In [ ]:
def get_ollama_model_names():
    with urllib.request.urlopen(f"{OLLAMA_BASE_URL}/api/tags", timeout=10) as response:
        payload = json.loads(response.read().decode("utf-8"))
    names = set()
    for item in payload.get("models", []):
        for key in ("name", "model"):
            value = item.get(key)
            if value:
                names.add(value)
    return names


def pull_ollama_model(model_name: str) -> bool:
    before = get_ollama_model_names()
    if model_name in before:
        print(f"Ollama model already available: {model_name}")
        return True

    print(f"Pulling Ollama VLM model: {model_name}")
    result = run(["ollama", "pull", model_name], check=False)
    if result.returncode != 0:
        return False

    after = get_ollama_model_names()
    if model_name in after:
        print(f"Ollama model downloaded: {model_name}")
    else:
        print(f"Ollama pull completed. Available models: {sorted(after)}")
    return True


if VLM_PROVIDER == "ollama":
    configure_ollama_environment()
    wait_for_ollama()

    selected_model = OLLAMA_VLM_MODEL
    if not pull_ollama_model(selected_model):
        print(f"Primary Ollama VLM pull failed: {selected_model}")
        print(f"Trying fallback Ollama VLM: {OLLAMA_VLM_FALLBACK_MODEL}")
        selected_model = OLLAMA_VLM_FALLBACK_MODEL
        if not pull_ollama_model(selected_model):
            raise RuntimeError(
                f"Could not pull either Ollama VLM model: {OLLAMA_VLM_MODEL}, {OLLAMA_VLM_FALLBACK_MODEL}"
            )

    OLLAMA_VLM_MODEL = selected_model
    os.environ["OLLAMA_VLM_MODEL"] = OLLAMA_VLM_MODEL
    run(["ollama", "list"], check=False)
    print("Selected Ollama VLM model for the pipeline:", OLLAMA_VLM_MODEL)
else:
    print("Ollama model download skipped because VLM_PROVIDER is not 'ollama'.")


## 7. Install SAM2 And SAMURAI Assets

Crowded-human mode uses YOLO + BoT-SORT/ReID for identity and SAM2 image prediction for masks. The SAMURAI repo is also cloned as a fallback for non-crowded object tracking and for crowded-human fallback cases.


In [ ]:
if not SAM2_REPO.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/facebookresearch/sam2.git", str(SAM2_REPO)])
else:
    print("SAM2 repo already exists:", SAM2_REPO)

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(SAM2_REPO)])


def patch_sam2_image_predictor_for_colab():
    """Fix upstream SAM2 .view(...) usage that can fail on Colab/PyTorch non-contiguous tensors."""
    predictor_path = SAM2_REPO / "sam2" / "sam2_image_predictor.py"
    if not predictor_path.exists():
        raise FileNotFoundError(f"SAM2 image predictor not found: {predictor_path}")
    old = "feat.permute(1, 2, 0).view(1, -1, *feat_size)"
    new = "feat.permute(1, 2, 0).reshape(1, -1, *feat_size)"
    text = predictor_path.read_text()
    if old in text:
        predictor_path.write_text(text.replace(old, new))
        print("Patched SAM2 image predictor for Colab/PyTorch compatibility:", predictor_path)
    elif new in text:
        print("SAM2 image predictor compatibility patch already applied:", predictor_path)
    else:
        print("SAM2 image predictor patch pattern not found; leaving file unchanged:", predictor_path)


patch_sam2_image_predictor_for_colab()

checkpoint_dir = SAM2_REPO / "checkpoints"
checkpoint_path = checkpoint_dir / "sam2.1_hiera_tiny.pt"
if not checkpoint_path.exists():
    run(["bash", "download_ckpts.sh"], cwd=checkpoint_dir)
else:
    print("SAM2 checkpoint already exists:", checkpoint_path)

if not SAMURAI_REPO.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/yangchris11/samurai.git", str(SAMURAI_REPO)])
else:
    print("SAMURAI repo already exists:", SAMURAI_REPO)

os.environ["SAM2_CHECKPOINT_PATH"] = str(checkpoint_path)
os.environ["SAM2_DEVICE"] = "cuda"
os.environ["SAM2_IMAGE_SIZE"] = "1024"
os.environ["CROWDED_HUMAN_SAM2_IMAGE_SIZE"] = CROWDED_HUMAN_SAM2_IMAGE_SIZE
os.environ["SAM2_TRACKING_BACKEND"] = "samurai"
os.environ["SAM2_BIDIRECTIONAL"] = "1"
os.environ["SAMURAI_REPO_DIR"] = str(SAMURAI_REPO)
os.environ["SAM2_MASK_CACHE_DIR"] = str(COLAB_OUTPUT_ROOT / "mask_cache")

print(json.dumps({
    "SAM2_CHECKPOINT_PATH": os.environ["SAM2_CHECKPOINT_PATH"],
    "SAM2_MODEL_CFG": os.environ.get("SAM2_MODEL_CFG", "<auto by backend>"),
    "SAM2_TRACKING_BACKEND": os.environ["SAM2_TRACKING_BACKEND"],
    "SAM2_IMAGE_SIZE": os.environ["SAM2_IMAGE_SIZE"],
    "CROWDED_HUMAN_SAM2_IMAGE_SIZE": os.environ["CROWDED_HUMAN_SAM2_IMAGE_SIZE"],
    "SAMURAI_REPO_DIR": os.environ["SAMURAI_REPO_DIR"],
}, indent=2))


## 8. Install VOID And Download VOID Models

This is the slowest setup step. It clones Netflix VOID, installs its runtime requirements, patches the inference script to honor the notebook's L4 profile flags, and downloads the CogVideoX base model plus `void_pass1.safetensors`.


In [ ]:
os.environ["VOID_REPO_DIR"] = str(VOID_REPO)
os.environ["VOID_REPO"] = str(VOID_REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if not VOID_REPO.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/Netflix/void-model.git", str(VOID_REPO)])
else:
    print("VOID repo already exists:", VOID_REPO)

run([sys.executable, "-m", "pip", "install", "-q", "-c", str(COLAB_CONSTRAINTS), "-r", "requirements.txt"], cwd=VOID_REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-c", str(COLAB_CONSTRAINTS), "huggingface_hub", "accelerate", "safetensors"], cwd=VOID_REPO)
run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall", "numpy==1.26.4", "opencv-python==4.10.0.84", "opencv-python-headless==4.10.0.84"])

predict_script = VOID_REPO / "inference" / "cogvideox_fun" / "predict_v2v.py"
text = predict_script.read_text()
old = "num_inference_steps = 30,"
new = "num_inference_steps = config.video_model.num_inference_steps,"
if old in text:
    text = text.replace(old, new)
    print("Patched VOID step count override.")
old = "            stack_mask = config.video_model.stack_mask,\n        ).videos"
new = "            stack_mask = config.video_model.stack_mask,\n            temporal_multidiffusion_stride = config.video_model.temproal_multidiffusion_stride,\n        ).videos"
if old in text:
    text = text.replace(old, new)
    print("Patched VOID temporal multidiffusion stride override.")
predict_script.write_text(text)

base_model_dir = VOID_REPO / "CogVideoX-Fun-V1.5-5b-InP"
if not base_model_dir.exists():
    run(["hf", "download", "alibaba-pai/CogVideoX-Fun-V1.5-5b-InP", "--local-dir", str(base_model_dir)], cwd=VOID_REPO)
else:
    print("CogVideoX base model already exists:", base_model_dir)

pass1_ckpt = VOID_REPO / "void_pass1.safetensors"
if not pass1_ckpt.exists():
    run(["hf", "download", "netflix/void-model", "void_pass1.safetensors", "--local-dir", str(VOID_REPO)], cwd=VOID_REPO)
else:
    print("VOID Pass 1 checkpoint already exists:", pass1_ckpt)

print("VOID runtime ready:", VOID_REPO)


## 9. Configure Selected Quality Restoration Backend

Real-ESRGAN is installed and wired automatically when selected. VEnhancer is wired through a wrapper: use `VENHANCER_FRAME_COMMAND_TEMPLATE` or `VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE` for adaptive parallel frame restoration, or `VENHANCER_INFERENCE_COMMAND_TEMPLATE` as a whole-video fallback when your VEnhancer checkout only supports video input.


In [ ]:
def write_venhancer_restore_wrapper():
    wrapper_source = r"""
import argparse
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
import threading
from pathlib import Path


def _print_process_line(prefix, line):
    if prefix:
        print(f"[{prefix}] {line}", flush=True)
    else:
        print(line, flush=True)


def _read_process_output(stream, prefix, tail_lines):
    buffer = []
    last_printed = ""

    def flush_buffer():
        nonlocal last_printed
        line = "".join(buffer).strip()
        buffer.clear()
        if not line or line == last_printed:
            return
        last_printed = line
        tail_lines.append(line)
        while len("\n".join(tail_lines)) > 4000 and len(tail_lines) > 1:
            tail_lines.pop(0)
        _print_process_line(prefix, line)

    while True:
        chunk = stream.read(1)
        if chunk == "":
            break
        if chunk in {"\n", "\r"}:
            flush_buffer()
        else:
            buffer.append(chunk)
    flush_buffer()


def start_shell(command, cwd=None, prefix=None):
    _print_process_line(prefix, "$ " + command)
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail_lines = []
    if process.stdout is None:
        return process, None, tail_lines, command
    thread = threading.Thread(
        target=_read_process_output,
        args=(process.stdout, prefix, tail_lines),
        daemon=True,
    )
    thread.start()
    return process, thread, tail_lines, command


def finish_process(process, thread, tail_lines, command, check=True):
    returncode = process.wait()
    if thread is not None:
        thread.join()
    output_tail = "\n".join(tail_lines)
    if check and returncode != 0:
        raise subprocess.CalledProcessError(returncode, command, output=output_tail, stderr=output_tail)
    return subprocess.CompletedProcess(command, returncode, stdout=output_tail, stderr="")


def run_shell(command, cwd=None, check=True, prefix=None):
    process, thread, tail_lines, rendered = start_shell(command, cwd=cwd, prefix=prefix)
    return finish_process(process, thread, tail_lines, rendered, check=check)


def run_args(args, check=True):
    printable = " ".join(shlex.quote(str(part)) for part in args)
    print("$", printable, flush=True)
    process = subprocess.run(args, check=False)
    if check and process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, args)
    return process.returncode


def frame_sort_key(path: Path):
    digits = []
    for char in path.stem:
        if char.isdigit():
            digits.append(char)
        elif digits:
            break
    return int("".join(digits) or "0")


def collect_images(path: Path):
    images = []
    for pattern in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
        images.extend(path.rglob(pattern))
    return sorted(images, key=frame_sort_key)


def split_evenly(items, worker_count):
    worker_count = max(1, min(worker_count, len(items)))
    chunk_size = (len(items) + worker_count - 1) // worker_count
    return [items[index:index + chunk_size] for index in range(0, len(items), chunk_size)]


def worker_retry_plan(requested_workers):
    requested_workers = max(1, int(requested_workers))
    candidates = [requested_workers, 4, 2, 1]
    plan = []
    for candidate in candidates:
        candidate = max(1, int(candidate))
        if candidate <= requested_workers and candidate not in plan:
            plan.append(candidate)
    if 1 not in plan:
        plan.append(1)
    return plan


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def normalize_video(source: Path, output: Path, fps: str, width: int, height: int, frame_count: int):
    if not source.exists():
        raise FileNotFoundError(f"VEnhancer command did not produce raw output: {source}")
    output.parent.mkdir(parents=True, exist_ok=True)
    normalized = output.with_name(output.stem + "_normalized" + output.suffix)
    if normalized.exists():
        normalized.unlink()
    run_args([
        "ffmpeg",
        "-y",
        "-hide_banner",
        "-loglevel",
        "error",
        "-i",
        str(source),
        "-map",
        "0:v:0",
        "-an",
        "-vf",
        f"setpts=PTS-STARTPTS,scale={width}:{height}:flags=lanczos,fps={fps}",
        "-frames:v",
        str(frame_count),
        "-c:v",
        "libx264",
        "-preset",
        "medium",
        "-crf",
        "18",
        "-pix_fmt",
        "yuv420p",
        "-movflags",
        "+faststart",
        str(normalized),
    ])
    shutil.move(str(normalized), str(output))


def extract_frames(input_path: str, frames_dir: Path):
    run_args([
        "ffmpeg",
        "-y",
        "-hide_banner",
        "-loglevel",
        "error",
        "-i",
        input_path,
        "-map",
        "0:v:0",
        "-vsync",
        "0",
        str(frames_dir / "%08d.png"),
    ])
    frames = sorted(frames_dir.glob("*.png"), key=frame_sort_key)
    if not frames:
        raise RuntimeError(f"No frames were extracted from {input_path}")
    print(f"Extracted {len(frames)} frames for VEnhancer frame restoration.", flush=True)
    return frames


def build_video_from_frames(sequence_dir: Path, output_path: Path, fps: str, width: int, height: int, frame_count: int):
    run_args([
        "ffmpeg",
        "-y",
        "-hide_banner",
        "-loglevel",
        "error",
        "-framerate",
        fps,
        "-i",
        str(sequence_dir / "%08d.png"),
        "-frames:v",
        str(frame_count),
        "-vf",
        f"scale={width}:{height}:flags=lanczos,fps={fps}",
        "-c:v",
        "libx264",
        "-preset",
        "medium",
        "-crf",
        "18",
        "-pix_fmt",
        "yuv420p",
        "-movflags",
        "+faststart",
        str(output_path),
    ])


def render_template(template: str, **values):
    return template.format(**{key: str(value) for key, value in values.items()})


def run_batch_worker(args, repo: Path, input_dir: Path, output_dir: Path, worker_index: int, worker_total: int):
    command = render_template(
        args.frame_batch_command_template,
        input_dir=input_dir,
        output_dir=output_dir,
        repo=repo,
        fps=args.fps,
        width=args.width,
        height=args.height,
        frame_count=args.frame_count,
    )
    run_shell(command, cwd=repo, prefix=f"worker {worker_index}/{worker_total}")


def run_frame_worker(args, repo: Path, frames, output_dir: Path, worker_index: int, worker_total: int):
    for frame_path in frames:
        output_path = output_dir / frame_path.name
        command = render_template(
            args.frame_command_template,
            input=frame_path,
            output=output_path,
            frame=frame_path,
            frame_index=frame_sort_key(frame_path),
            output_dir=output_dir,
            repo=repo,
            fps=args.fps,
            width=args.width,
            height=args.height,
            frame_count=args.frame_count,
        )
        run_shell(command, cwd=repo, prefix=f"worker {worker_index}/{worker_total}")
        if not output_path.exists():
            raise FileNotFoundError(f"VEnhancer frame command did not write {output_path}")


def run_venhancer_workers(args, repo: Path, frames, frames_dir: Path, attempt_dir: Path, worker_count: int):
    chunks = split_evenly(frames, worker_count)
    input_root = attempt_dir / "worker_inputs"
    output_root = attempt_dir / "worker_outputs"
    reset_dir(input_root)
    reset_dir(output_root)
    processes = []
    failures = []
    lock = threading.Lock()

    print(f"Trying VEnhancer frame restoration with {len(chunks)} worker(s).", flush=True)

    def worker_main(index, chunk):
        input_dir = input_root / f"worker_{index:03d}"
        output_dir = output_root / f"worker_{index:03d}"
        input_dir.mkdir(parents=True, exist_ok=True)
        output_dir.mkdir(parents=True, exist_ok=True)
        for frame_path in chunk:
            shutil.copyfile(frame_path, input_dir / frame_path.name)
        print(f"VEnhancer worker {index}/{len(chunks)} assigned {len(chunk)} frame(s).", flush=True)
        try:
            if args.frame_batch_command_template.strip():
                run_batch_worker(args, repo, input_dir, output_dir, index, len(chunks))
            else:
                worker_frames = sorted(input_dir.glob("*.png"), key=frame_sort_key)
                run_frame_worker(args, repo, worker_frames, output_dir, index, len(chunks))
        except Exception as exc:
            with lock:
                failures.append((index, exc))

    threads = []
    for index, chunk in enumerate(chunks, start=1):
        thread = threading.Thread(target=worker_main, args=(index, chunk), daemon=True)
        thread.start()
        threads.append(thread)
    for thread in threads:
        thread.join()

    if failures:
        details = "\n".join(f"worker {index}: {exc}" for index, exc in failures)
        raise RuntimeError(f"VEnhancer worker failure:\n{details}")

    output_dirs = sorted(output_root.glob("worker_*"))
    enhanced_count = sum(len(collect_images(output_dir)) for output_dir in output_dirs)
    if enhanced_count != len(frames):
        raise RuntimeError(f"VEnhancer produced {enhanced_count} frame(s), expected {len(frames)}")
    return output_dirs


def run_venhancer_workers_with_retry(args, repo: Path, frames, frames_dir: Path, temp_root: Path):
    last_error = None
    for worker_count in worker_retry_plan(args.workers):
        attempt_dir = temp_root / f"venhancer_attempt_workers_{worker_count}"
        reset_dir(attempt_dir)
        try:
            output_dirs = run_venhancer_workers(args, repo, frames, frames_dir, attempt_dir, worker_count)
            print(f"VEnhancer worker attempt succeeded with {worker_count} worker(s).", flush=True)
            return output_dirs
        except Exception as exc:
            last_error = exc
            print(f"VEnhancer worker attempt with {worker_count} worker(s) failed: {exc}", flush=True)
            if worker_count > 1:
                print("Cleaning partial VEnhancer outputs and retrying with fewer workers.", flush=True)
            reset_dir(attempt_dir)
    raise RuntimeError(f"All VEnhancer worker retry attempts failed. Last error: {last_error}")


def run_frame_restore(args, repo: Path, temp_root: Path):
    frames_dir = temp_root / "frames"
    sequence_dir = temp_root / "sequence_frames"
    frames_dir.mkdir(parents=True, exist_ok=True)
    sequence_dir.mkdir(parents=True, exist_ok=True)
    frames = extract_frames(args.input, frames_dir)
    output_dirs = run_venhancer_workers_with_retry(args, repo, frames, frames_dir, temp_root)

    enhanced_images = []
    for output_dir in output_dirs:
        enhanced_images.extend(collect_images(output_dir))
    enhanced_images = sorted(enhanced_images, key=frame_sort_key)
    if len(enhanced_images) != len(frames):
        raise RuntimeError(f"VEnhancer produced {len(enhanced_images)} frame(s), expected {len(frames)}")

    for index, image in enumerate(enhanced_images, start=1):
        shutil.copyfile(image, sequence_dir / f"{index:08d}.png")

    frame_video = temp_root / "venhancer_frame_restore.mp4"
    build_video_from_frames(sequence_dir, frame_video, args.fps, args.width, args.height, args.frame_count)
    return frame_video


def run_video_restore(args, repo: Path, temp_root: Path):
    template = args.video_command_template.strip()
    if not template:
        raise RuntimeError(
            "No VEnhancer frame or video command template is configured. Set one of: "
            "VENHANCER_FRAME_COMMAND_TEMPLATE, VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE, "
            "or VENHANCER_INFERENCE_COMMAND_TEMPLATE."
        )
    raw_output = temp_root / "venhancer_raw.mp4"
    command = render_template(
        template,
        input=Path(args.input),
        raw_output=raw_output,
        output=raw_output,
        fps=args.fps,
        width=args.width,
        height=args.height,
        frame_count=args.frame_count,
        repo=repo,
    )
    run_shell(command, cwd=repo)
    return raw_output


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--fps", required=True)
    parser.add_argument("--width", required=True, type=int)
    parser.add_argument("--height", required=True, type=int)
    parser.add_argument("--frame-count", required=True, type=int)
    parser.add_argument("--repo", default=os.environ.get("VENHANCER_REPO", "/content/VEnhancer"))
    parser.add_argument("--workers", default=os.environ.get("VENHANCER_WORKERS", "4"))
    parser.add_argument("--frame-command-template", default=os.environ.get("VENHANCER_FRAME_COMMAND_TEMPLATE", ""))
    parser.add_argument("--frame-batch-command-template", default=os.environ.get("VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE", ""))
    parser.add_argument("--video-command-template", default=os.environ.get("VENHANCER_INFERENCE_COMMAND_TEMPLATE", ""))
    args = parser.parse_args()

    repo = Path(args.repo)
    temp_root = Path(tempfile.mkdtemp(prefix="venhancer-restore-"))
    try:
        if args.frame_batch_command_template.strip() or args.frame_command_template.strip():
            print(f"Using VEnhancer adaptive frame restoration. Retry plan: {worker_retry_plan(args.workers)}", flush=True)
            raw_output = run_frame_restore(args, repo, temp_root)
        else:
            print("Using VEnhancer whole-video fallback. Frame workers are disabled until a frame command template is set.", flush=True)
            raw_output = run_video_restore(args, repo, temp_root)
        normalize_video(raw_output, Path(args.output), args.fps, args.width, args.height, args.frame_count)
        print("VEnhancer restored output:", args.output, flush=True)
    finally:
        shutil.rmtree(temp_root, ignore_errors=True)


if __name__ == "__main__":
    main()
"""
    VENHANCER_WRAPPER.write_text(wrapper_source)
    print("Wrote VEnhancer wrapper:", VENHANCER_WRAPPER)


def install_venhancer_for_colab():
    if not INSTALL_VENHANCER:
        print("VEnhancer repo install skipped because INSTALL_VENHANCER is False.")
        return
    if not VENHANCER_REPO.exists():
        run(["git", "clone", "--depth", "1", VENHANCER_GIT_URL, str(VENHANCER_REPO)])
    else:
        print("VEnhancer repo already exists:", VENHANCER_REPO)
    requirements = VENHANCER_REPO / "requirements.txt"
    if requirements.exists():
        pip_install(["-q", "-c", str(COLAB_CONSTRAINTS), "-r", str(requirements)])
    else:
        print("No VEnhancer requirements.txt found; install dependencies manually for this checkout:", VENHANCER_REPO)


def set_venhancer_template():
    os.environ["VENHANCER_REPO"] = str(VENHANCER_REPO)
    os.environ["VENHANCER_WORKERS"] = VENHANCER_WORKERS
    os.environ["VENHANCER_INFERENCE_COMMAND_TEMPLATE"] = VENHANCER_INFERENCE_COMMAND_TEMPLATE.strip()
    os.environ["VENHANCER_FRAME_COMMAND_TEMPLATE"] = VENHANCER_FRAME_COMMAND_TEMPLATE.strip()
    os.environ["VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE"] = VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE.strip()
    os.environ["VENHANCER_COMMAND_TEMPLATE"] = (
        f"{sys.executable} {VENHANCER_WRAPPER} "
        "--input \"{input}\" --output \"{output}\" --fps {fps} "
        "--width {width} --height {height} --frame-count {frame_count} "
        f"--repo {VENHANCER_REPO} --workers {VENHANCER_WORKERS}"
    )
    print("VENHANCER_COMMAND_TEMPLATE:", os.environ["VENHANCER_COMMAND_TEMPLATE"])


def configure_venhancer_for_colab():
    has_video = bool(VENHANCER_INFERENCE_COMMAND_TEMPLATE.strip())
    has_frame = bool(VENHANCER_FRAME_COMMAND_TEMPLATE.strip() or VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE.strip())
    if not has_video and not has_frame:
        raise RuntimeError(
            "QUALITY_RESTORATION='venhancer' requires one VEnhancer command template. "
            "For parallel frame workers set VENHANCER_FRAME_COMMAND_TEMPLATE or "
            "VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE. For whole-video fallback set "
            "VENHANCER_INFERENCE_COMMAND_TEMPLATE."
        )
    if has_video and not has_frame:
        print("VEnhancer is configured with whole-video fallback only; frame workers are disabled.")
    if has_frame:
        print(f"VEnhancer frame workers enabled with retry plan from {VENHANCER_WORKERS} workers.")
    install_venhancer_for_colab()
    write_venhancer_restore_wrapper()
    set_venhancer_template()


In [ ]:
def pip_install(args, check=True):
    return run([sys.executable, "-m", "pip", "install", *args], check=check)


def patch_basicsr_for_current_torchvision():
    """Real-ESRGAN's basicsr dependency may import a removed torchvision module."""
    import site

    patched = False
    for site_dir in site.getsitepackages():
        degradations_path = Path(site_dir) / "basicsr" / "data" / "degradations.py"
        if not degradations_path.exists():
            continue
        text = degradations_path.read_text()
        old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
        new = "from torchvision.transforms.functional import rgb_to_grayscale"
        if old in text:
            degradations_path.write_text(text.replace(old, new))
            patched = True
            print("Patched basicsr torchvision import:", degradations_path)
    if not patched:
        print("No basicsr torchvision compatibility patch was needed.")


def write_realesrgan_restore_wrapper():
    wrapper_source = 'import argparse\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport threading\nfrom pathlib import Path\n\n\ndef _print_process_line(prefix, line):\n    if prefix:\n        print(f"[{prefix}] {line}", flush=True)\n    else:\n        print(line, flush=True)\n\n\ndef _read_process_output(stream, prefix, tail_lines):\n    buffer = []\n    last_printed = ""\n\n    def flush_buffer():\n        nonlocal last_printed\n        line = "".join(buffer).strip()\n        buffer.clear()\n        if not line or line == last_printed:\n            return\n        last_printed = line\n        tail_lines.append(line)\n        while len("\\n".join(tail_lines)) > 4000 and len(tail_lines) > 1:\n            tail_lines.pop(0)\n        _print_process_line(prefix, line)\n\n    while True:\n        chunk = stream.read(1)\n        if chunk == "":\n            break\n        if chunk in {"\\n", "\\r"}:\n            flush_buffer()\n        else:\n            buffer.append(chunk)\n    flush_buffer()\n\n\ndef start_process(cmd, cwd=None, prefix=None):\n    cmd = [str(part) for part in cmd]\n    printable = "$ " + " ".join(cmd)\n    _print_process_line(prefix, printable)\n    process = subprocess.Popen(\n        cmd,\n        cwd=str(cwd) if cwd else None,\n        stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT,\n        text=True,\n        bufsize=1,\n    )\n    tail_lines = []\n    if process.stdout is None:\n        return process, None, tail_lines, cmd\n    thread = threading.Thread(\n        target=_read_process_output,\n        args=(process.stdout, prefix, tail_lines),\n        daemon=True,\n    )\n    thread.start()\n    return process, thread, tail_lines, cmd\n\n\ndef finish_process(process, thread, tail_lines, cmd, check=True):\n    returncode = process.wait()\n    if thread is not None:\n        thread.join()\n    output_tail = "\\n".join(tail_lines)\n    if check and returncode != 0:\n        raise subprocess.CalledProcessError(\n            returncode,\n            cmd,\n            output=output_tail,\n            stderr=output_tail,\n        )\n    return subprocess.CompletedProcess(cmd, returncode, stdout=output_tail, stderr="")\n\n\ndef run(cmd, cwd=None, check=True, prefix=None):\n    process, thread, tail_lines, command = start_process(cmd, cwd=cwd, prefix=prefix)\n    return finish_process(process, thread, tail_lines, command, check=check)\n\n\ndef newest_video(path: Path):\n    candidates = []\n    for pattern in ("*.mp4", "*.mov", "*.mkv", "*.avi"):\n        candidates.extend(path.rglob(pattern))\n    if not candidates:\n        return None\n    return max(candidates, key=lambda item: item.stat().st_mtime)\n\n\ndef frame_sort_key(path: Path):\n    digits = []\n    for char in path.stem:\n        if char.isdigit():\n            digits.append(char)\n        elif digits:\n            break\n    return int("".join(digits) or "0")\n\n\ndef collect_images(path: Path):\n    images = []\n    for pattern in ("*.png", "*.jpg", "*.jpeg", "*.webp"):\n        images.extend(path.rglob(pattern))\n    restored = [item for item in images if "restored" in item.stem]\n    return sorted(restored or images, key=frame_sort_key)\n\n\ndef split_evenly(items, worker_count):\n    worker_count = max(1, min(worker_count, len(items)))\n    chunk_size = (len(items) + worker_count - 1) // worker_count\n    return [items[index:index + chunk_size] for index in range(0, len(items), chunk_size)]\n\n\ndef worker_retry_plan(requested_workers):\n    requested_workers = max(1, int(requested_workers))\n    candidates = [requested_workers, 6, 4, 2, 1]\n    plan = []\n    for candidate in candidates:\n        candidate = max(1, int(candidate))\n        if candidate <= requested_workers and candidate not in plan:\n            plan.append(candidate)\n    if 1 not in plan:\n        plan.append(1)\n    return plan\n\n\ndef reset_dir(path: Path):\n    if path.exists():\n        shutil.rmtree(path)\n    path.mkdir(parents=True, exist_ok=True)\n\n\ndef normalize_video(source: Path, output: Path, fps: str, width: int, height: int, frame_count: int):\n    output.parent.mkdir(parents=True, exist_ok=True)\n    normalized = output.with_name(output.stem + "_normalized" + output.suffix)\n    if normalized.exists():\n        normalized.unlink()\n    run([\n        "ffmpeg",\n        "-y",\n        "-hide_banner",\n        "-loglevel",\n        "error",\n        "-i",\n        str(source),\n        "-map",\n        "0:v:0",\n        "-an",\n        "-vf",\n        f"setpts=PTS-STARTPTS,scale={width}:{height}:flags=lanczos,fps={fps}",\n        "-frames:v",\n        str(frame_count),\n        "-c:v",\n        "libx264",\n        "-preset",\n        "medium",\n        "-crf",\n        "18",\n        "-pix_fmt",\n        "yuv420p",\n        "-movflags",\n        "+faststart",\n        str(normalized),\n    ])\n    shutil.move(str(normalized), str(output))\n\n\ndef try_video_inference(args, repo: Path, temp_root: Path):\n    inference_script = repo / "inference_realesrgan_video.py"\n    if not inference_script.exists():\n        print(f"Real-ESRGAN video script not found, using frame fallback: {inference_script}", flush=True)\n        return None\n\n    video_dir = temp_root / "video_script"\n    video_dir.mkdir(parents=True, exist_ok=True)\n    result = run([\n        sys.executable,\n        str(inference_script),\n        "-i",\n        args.input,\n        "-n",\n        args.model,\n        "-o",\n        str(video_dir),\n        "--outscale",\n        str(args.outscale),\n        "--suffix",\n        "restored",\n        "--tile",\n        str(args.tile),\n        "--ffmpeg_bin",\n        "ffmpeg",\n    ], cwd=repo, check=False)\n    if result.returncode != 0:\n        print(\n            f"Real-ESRGAN video script failed with exit code {result.returncode}; switching to frame fallback.",\n            flush=True,\n        )\n        return None\n\n    enhanced = newest_video(video_dir)\n    if enhanced is None:\n        print(f"Real-ESRGAN video script produced no video under {video_dir}; switching to frame fallback.", flush=True)\n        return None\n    return enhanced\n\n\ndef run_realesrgan_workers(args, repo: Path, inference_script: Path, extracted, frames_dir: Path, attempt_dir: Path, worker_count: int):\n    chunks = split_evenly(extracted, worker_count)\n    enhanced_root = attempt_dir / "enhanced_frames"\n    reset_dir(enhanced_root)\n\n    if len(chunks) == 1:\n        enhanced_dir = enhanced_root / "worker_001"\n        enhanced_dir.mkdir(parents=True, exist_ok=True)\n        run([\n            sys.executable,\n            str(inference_script),\n            "-i",\n            str(frames_dir),\n            "-n",\n            args.model,\n            "-o",\n            str(enhanced_dir),\n            "--outscale",\n            str(args.outscale),\n            "--suffix",\n            "restored",\n            "--tile",\n            str(args.tile),\n        ], cwd=repo, prefix="worker 1/1")\n        return [enhanced_dir]\n\n    worker_inputs_root = attempt_dir / "worker_inputs"\n    reset_dir(worker_inputs_root)\n    processes = []\n    output_dirs = []\n    print(\n        f"Trying Real-ESRGAN frame fallback with {len(chunks)} parallel workers, tile={args.tile}.",\n        flush=True,\n    )\n\n    for index, chunk in enumerate(chunks, start=1):\n        input_dir = worker_inputs_root / f"worker_{index:03d}"\n        output_dir = enhanced_root / f"worker_{index:03d}"\n        input_dir.mkdir(parents=True, exist_ok=True)\n        output_dir.mkdir(parents=True, exist_ok=True)\n        for frame_path in chunk:\n            shutil.copyfile(frame_path, input_dir / frame_path.name)\n        print(f"Worker {index}/{len(chunks)} assigned {len(chunk)} frames.", flush=True)\n        process, thread, tail_lines, command = start_process([\n            sys.executable,\n            str(inference_script),\n            "-i",\n            str(input_dir),\n            "-n",\n            args.model,\n            "-o",\n            str(output_dir),\n            "--outscale",\n            str(args.outscale),\n            "--suffix",\n            "restored",\n            "--tile",\n            str(args.tile),\n        ], cwd=repo, prefix=f"worker {index}/{len(chunks)}")\n        processes.append((process, thread, tail_lines, command, index))\n        output_dirs.append(output_dir)\n\n    failures = []\n    for process, thread, tail_lines, command, index in processes:\n        result = finish_process(process, thread, tail_lines, command, check=False)\n        if result.returncode != 0:\n            failures.append((index, result.returncode, result.stdout))\n    if failures:\n        details = "\\n".join(\n            f"worker {index} exited {returncode}: {tail}" for index, returncode, tail in failures\n        )\n        raise RuntimeError(f"Parallel Real-ESRGAN worker failure:\\n{details}")\n    return output_dirs\n\n\ndef run_realesrgan_workers_with_retry(args, repo: Path, inference_script: Path, extracted, frames_dir: Path, temp_root: Path):\n    last_error = None\n    for worker_count in worker_retry_plan(args.workers):\n        attempt_dir = temp_root / f"attempt_workers_{worker_count}"\n        reset_dir(attempt_dir)\n        try:\n            output_dirs = run_realesrgan_workers(\n                args,\n                repo,\n                inference_script,\n                extracted,\n                frames_dir,\n                attempt_dir,\n                worker_count,\n            )\n            print(f"Real-ESRGAN worker attempt succeeded with {worker_count} worker(s).", flush=True)\n            return output_dirs\n        except Exception as exc:\n            last_error = exc\n            print(\n                f"Real-ESRGAN worker attempt with {worker_count} worker(s) failed: {exc}",\n                flush=True,\n            )\n            if worker_count > 1:\n                print("Cleaning partial outputs and retrying with fewer workers.", flush=True)\n            reset_dir(attempt_dir)\n    raise RuntimeError(f"All Real-ESRGAN worker retry attempts failed. Last error: {last_error}")\n\n\ndef run_frame_inference(args, repo: Path, temp_root: Path):\n    inference_script = repo / "inference_realesrgan.py"\n    if not inference_script.exists():\n        raise FileNotFoundError(f"Real-ESRGAN image inference script not found: {inference_script}")\n\n    frames_dir = temp_root / "frames"\n    sequence_dir = temp_root / "sequence_frames"\n    frames_dir.mkdir(parents=True, exist_ok=True)\n    sequence_dir.mkdir(parents=True, exist_ok=True)\n\n    run([\n        "ffmpeg",\n        "-y",\n        "-hide_banner",\n        "-loglevel",\n        "error",\n        "-i",\n        args.input,\n        "-map",\n        "0:v:0",\n        "-vsync",\n        "0",\n        str(frames_dir / "%08d.png"),\n    ])\n\n    extracted = sorted(frames_dir.glob("*.png"), key=frame_sort_key)\n    if not extracted:\n        raise RuntimeError(f"No frames were extracted from {args.input}")\n    print(f"Extracted {len(extracted)} frames for Real-ESRGAN frame fallback.", flush=True)\n\n    output_dirs = run_realesrgan_workers_with_retry(args, repo, inference_script, extracted, frames_dir, temp_root)\n    enhanced_images = []\n    for output_dir in output_dirs:\n        enhanced_images.extend(collect_images(output_dir))\n    enhanced_images = sorted(enhanced_images, key=frame_sort_key)\n\n    if not enhanced_images:\n        raise RuntimeError(f"Real-ESRGAN frame fallback produced no images under {temp_root}")\n    if len(enhanced_images) != len(extracted):\n        print(\n            f"Warning: extracted {len(extracted)} frames but enhanced {len(enhanced_images)} frames.",\n            flush=True,\n        )\n\n    for index, image in enumerate(enhanced_images, start=1):\n        shutil.copyfile(image, sequence_dir / f"{index:08d}.png")\n\n    intermediate = temp_root / "realesrgan_frame_fallback.mp4"\n    run([\n        "ffmpeg",\n        "-y",\n        "-hide_banner",\n        "-loglevel",\n        "error",\n        "-framerate",\n        args.fps,\n        "-i",\n        str(sequence_dir / "%08d.png"),\n        "-frames:v",\n        str(args.frame_count),\n        "-vf",\n        f"scale={args.width}:{args.height}:flags=lanczos,fps={args.fps}",\n        "-c:v",\n        "libx264",\n        "-preset",\n        "medium",\n        "-crf",\n        "18",\n        "-pix_fmt",\n        "yuv420p",\n        "-movflags",\n        "+faststart",\n        str(intermediate),\n    ])\n    return intermediate\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", required=True)\n    parser.add_argument("--output", required=True)\n    parser.add_argument("--fps", required=True)\n    parser.add_argument("--width", required=True, type=int)\n    parser.add_argument("--height", required=True, type=int)\n    parser.add_argument("--frame-count", required=True, type=int)\n    parser.add_argument("--repo", default="/content/Real-ESRGAN")\n    parser.add_argument("--model", default="RealESRGAN_x4plus")\n    parser.add_argument("--outscale", default="1")\n    parser.add_argument("--tile", default="128")\n    parser.add_argument("--workers", default="6")\n    args = parser.parse_args()\n\n    repo = Path(args.repo)\n    if not repo.exists():\n        raise FileNotFoundError(f"Real-ESRGAN repo not found: {repo}")\n\n    worker_count = max(1, int(args.workers))\n    temp_root = Path(tempfile.mkdtemp(prefix="realesrgan-restore-"))\n    try:\n        if worker_count > 1:\n            print(\n                f"Skipping Real-ESRGAN video script because workers={worker_count}; using adaptive parallel frame fallback.",\n                flush=True,\n            )\n            print(f"Real-ESRGAN worker retry plan: {worker_retry_plan(worker_count)}", flush=True)\n            enhanced = run_frame_inference(args, repo, temp_root)\n        else:\n            enhanced = try_video_inference(args, repo, temp_root)\n            if enhanced is None:\n                enhanced = run_frame_inference(args, repo, temp_root)\n        output = Path(args.output)\n        normalize_video(enhanced, output, args.fps, args.width, args.height, args.frame_count)\n        print("Real-ESRGAN restored output:", output, flush=True)\n    finally:\n        shutil.rmtree(temp_root, ignore_errors=True)\n\n\nif __name__ == "__main__":\n    main()\n'
    REALESRGAN_WRAPPER.write_text(wrapper_source)
    print("Wrote Real-ESRGAN wrapper:", REALESRGAN_WRAPPER)


def set_realesrgan_template():
    os.environ["REALESRGAN_REPO"] = str(REALESRGAN_REPO)
    os.environ["REALESRGAN_MODEL"] = REALESRGAN_MODEL
    os.environ["REALESRGAN_OUTSCALE"] = REALESRGAN_OUTSCALE
    os.environ["REALESRGAN_TILE"] = REALESRGAN_TILE
    os.environ["REALESRGAN_WORKERS"] = REALESRGAN_WORKERS
    os.environ["REALESRGAN_COMMAND_TEMPLATE"] = (
        f"{sys.executable} {REALESRGAN_WRAPPER} "
        "--input \"{input}\" --output \"{output}\" --fps {fps} "
        "--width {width} --height {height} --frame-count {frame_count} "
        f"--repo {REALESRGAN_REPO} --model {REALESRGAN_MODEL} "
        f"--outscale {REALESRGAN_OUTSCALE} --tile {REALESRGAN_TILE} "
        f"--workers {REALESRGAN_WORKERS}"
    )
    print("REALESRGAN_COMMAND_TEMPLATE:", os.environ["REALESRGAN_COMMAND_TEMPLATE"])


def install_realesrgan_for_colab():
    if not INSTALL_REALESRGAN:
        print("Real-ESRGAN install skipped because INSTALL_REALESRGAN is False.")
        return
    if not REALESRGAN_REPO.exists():
        run(["git", "clone", "--depth", "1", "https://github.com/xinntao/Real-ESRGAN.git", str(REALESRGAN_REPO)])
    else:
        print("Real-ESRGAN repo already exists:", REALESRGAN_REPO)

    pip_install(["-q", "-c", str(COLAB_CONSTRAINTS), "basicsr", "facexlib", "gfpgan"])
    pip_install(["-q", "-c", str(COLAB_CONSTRAINTS), "-r", str(REALESRGAN_REPO / "requirements.txt")])
    pip_install(["-q", "--no-deps", "-e", str(REALESRGAN_REPO)])
    patch_basicsr_for_current_torchvision()

    weights_dir = REALESRGAN_REPO / "weights"
    weights_dir.mkdir(parents=True, exist_ok=True)
    weights_path = weights_dir / f"{REALESRGAN_MODEL}.pth"
    if not weights_path.exists():
        run([
            "curl",
            "-fL",
            f"https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/{REALESRGAN_MODEL}.pth",
            "-o",
            str(weights_path),
        ])
    else:
        print("Real-ESRGAN weights already exist:", weights_path)
    write_realesrgan_restore_wrapper()
    set_realesrgan_template()


def configure_quality_restoration_backend():
    os.environ["QUALITY_RESTORATION"] = QUALITY_RESTORATION
    if QUALITY_RESTORATION in {"none", "ffmpeg_bicubic"}:
        print("External quality restoration setup not needed for:", QUALITY_RESTORATION)
        return
    if QUALITY_RESTORATION == "realesrgan":
        install_realesrgan_for_colab()
        return
    if QUALITY_RESTORATION == "venhancer":
        configure_venhancer_for_colab()
        return
    raise RuntimeError(f"Unknown QUALITY_RESTORATION: {QUALITY_RESTORATION}")


configure_quality_restoration_backend()


## 10. Configure Phase 2 And Phase 3 Runtime Defaults

These environment variables are read by the app and Celery worker. They are tuned for Colab L4 and can be adjusted before launching the UI.


In [ ]:
def resolve_reid_model(default_name: str) -> str:
    """Use a valid local ReID checkpoint when present; otherwise download the BoxMOT registry file."""
    default_path = Path(default_name)
    if default_path.is_absolute() and default_path.exists() and default_path.stat().st_size > 1024 * 1024:
        return str(default_path)

    candidate_paths = [
        REID_WEIGHTS_DIR / default_name,
        PROJECT_ROOT / default_name,
        PROJECT_ROOT / "models" / default_name,
        Path("/content") / default_name,
        Path("/content/drive/MyDrive") / default_name,
        Path("/content/drive/MyDrive/LiveInterview") / default_name,
    ]
    for path in candidate_paths:
        if not path.exists():
            continue
        size = path.stat().st_size
        if size > 1024 * 1024:
            print(f"Using local ReID model: {path} ({size / 1024 / 1024:.1f} MB)")
            return str(path)
        print(f"Ignoring suspiciously small ReID file: {path} ({size} bytes)")

    url = CROWDED_HUMAN_REID_URLS.get(default_name)
    if url:
        target = REID_WEIGHTS_DIR / default_name
        try:
            import gdown

            print(f"Downloading ReID model with gdown: {default_name}")
            gdown.download(url=url, output=str(target), quiet=False, fuzzy=True)
            size = target.stat().st_size if target.exists() else 0
            if size > 1024 * 1024:
                print(f"Using downloaded ReID model: {target} ({size / 1024 / 1024:.1f} MB)")
                return str(target)
            print(f"Downloaded ReID file is too small, falling back to BoxMOT name: {target} ({size} bytes)")
        except Exception as exc:
            print(f"ReID prefetch failed ({exc}); BoxMOT will resolve by model name: {default_name}")
    else:
        print(f"No known prefetch URL for ReID model: {default_name}")
    print(f"No local ReID checkpoint found. BoxMOT will resolve by model name: {default_name}")
    return default_name


CROWDED_HUMAN_REID_MODEL = resolve_reid_model(CROWDED_HUMAN_REID_MODEL)

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["CROWDED_HUMAN_DETECTOR_MODEL"] = CROWDED_HUMAN_DETECTOR_MODEL
os.environ["CROWDED_HUMAN_REID_MODEL"] = CROWDED_HUMAN_REID_MODEL
os.environ["CROWDED_HUMAN_DETECTOR_IMGSZ"] = CROWDED_HUMAN_DETECTOR_IMGSZ
os.environ["CROWDED_HUMAN_PERSON_CONF"] = CROWDED_HUMAN_PERSON_CONF
os.environ["CROWDED_HUMAN_PERSON_IOU"] = CROWDED_HUMAN_PERSON_IOU
os.environ["CROWDED_HUMAN_TARGET_ROI_MIN_IOU"] = CROWDED_HUMAN_TARGET_ROI_MIN_IOU
os.environ["CROWDED_HUMAN_SAM2_IMAGE_SIZE"] = CROWDED_HUMAN_SAM2_IMAGE_SIZE
os.environ["CROWDED_HUMAN_SAM2_REFINE_EVERY_N_FRAMES"] = CROWDED_HUMAN_SAM2_REFINE_EVERY_N_FRAMES
os.environ["CROWDED_HUMAN_MISSING_TARGET_WARNING_FRAMES"] = CROWDED_HUMAN_MISSING_TARGET_WARNING_FRAMES
os.environ["CROWDED_HUMAN_NEIGHBOR_EXCLUSION"] = CROWDED_HUMAN_NEIGHBOR_EXCLUSION
os.environ["CROWDED_HUMAN_NEIGHBOR_MASK_EROSION_PX"] = CROWDED_HUMAN_NEIGHBOR_MASK_EROSION_PX

# FastAPI/Celery can pass these as form fields, but defaults are useful for CLI cells.
os.environ["RESOURCE_PROFILE"] = RESOURCE_PROFILE
os.environ["QUALITY_RESTORATION"] = QUALITY_RESTORATION
if QUALITY_RESTORATION == "realesrgan" and REALESRGAN_WRAPPER.exists():
    set_realesrgan_template()
elif QUALITY_RESTORATION == "venhancer" and VENHANCER_WRAPPER.exists():
    set_venhancer_template()
elif QUALITY_RESTORATION == "venhancer" and VENHANCER_COMMAND_TEMPLATE.strip():
    os.environ["VENHANCER_COMMAND_TEMPLATE"] = VENHANCER_COMMAND_TEMPLATE.strip()
os.environ["VLM_PROVIDER"] = VLM_PROVIDER
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
os.environ["OLLAMA_BASE_URL"] = OLLAMA_BASE_URL
os.environ["OLLAMA_VLM_MODEL"] = OLLAMA_VLM_MODEL
os.environ["OLLAMA_MODELS"] = str(OLLAMA_MODELS_DIR)
os.environ["OLLAMA_VLM_SAMPLE_FRAMES"] = OLLAMA_VLM_SAMPLE_FRAMES
os.environ["OLLAMA_VLM_MAX_IMAGE_SIDE"] = OLLAMA_VLM_MAX_IMAGE_SIDE
os.environ["OLLAMA_VLM_TIMEOUT_SECONDS"] = OLLAMA_VLM_TIMEOUT_SECONDS
os.environ["OLLAMA_VLM_KEEP_ALIVE"] = OLLAMA_VLM_KEEP_ALIVE

# Keep package caches and model configs in Colab-local storage for speed.
os.environ["YOLO_CONFIG_DIR"] = "/content/ultralytics_config"
os.environ["MPLCONFIGDIR"] = "/content/matplotlib_config"
Path(os.environ["YOLO_CONFIG_DIR"]).mkdir(parents=True, exist_ok=True)
(Path(os.environ["YOLO_CONFIG_DIR"]) / "Ultralytics").mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
for config_dir in (Path(os.environ["YOLO_CONFIG_DIR"]), Path(os.environ["YOLO_CONFIG_DIR"]) / "Ultralytics", Path(os.environ["MPLCONFIGDIR"])):
    try:
        config_dir.chmod(0o777)
    except OSError:
        pass

runtime_summary_keys = [
    "CROWDED_HUMAN_DETECTOR_MODEL",
    "CROWDED_HUMAN_REID_MODEL",
    "CROWDED_HUMAN_DETECTOR_IMGSZ",
    "CROWDED_HUMAN_PERSON_CONF",
    "CROWDED_HUMAN_PERSON_IOU",
    "CROWDED_HUMAN_TARGET_ROI_MIN_IOU",
    "CROWDED_HUMAN_SAM2_IMAGE_SIZE",
    "CROWDED_HUMAN_NEIGHBOR_EXCLUSION",
    "VOID_REPO_DIR",
    "SAM2_CHECKPOINT_PATH",
    "QUALITY_RESTORATION",
    "REALESRGAN_COMMAND_TEMPLATE",
    "VENHANCER_COMMAND_TEMPLATE",
    "VENHANCER_WORKERS",
    "VENHANCER_FRAME_COMMAND_TEMPLATE",
    "VENHANCER_FRAME_BATCH_COMMAND_TEMPLATE",
    "VENHANCER_INFERENCE_COMMAND_TEMPLATE",
    "VLM_PROVIDER",
    "OLLAMA_BASE_URL",
    "OLLAMA_VLM_MODEL",
    "OLLAMA_VLM_KEEP_ALIVE",
]
print(json.dumps({key: os.environ[key] for key in runtime_summary_keys if key in os.environ}, indent=2))


## 11. Preflight Imports And Model Warm-Up

This validates the Python environment before spending time in the UI. YOLO weight download may happen here on first run.


In [ ]:
import os
import sys
import importlib
import importlib.metadata as metadata
from pathlib import Path

os.chdir(PROJECT_ROOT)

# Remove Colab's /content scratch path if it shadows installed packages.
if "/content" in sys.path:
    sys.path.remove("/content")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
if str(SAM2_REPO) not in sys.path:
    sys.path.insert(0, str(SAM2_REPO))

import fastapi
import uvicorn
import cv2
import numpy as np

# Fail early if Colab still has mixed NumPy binaries from an earlier install.
print("NumPy:", np.__version__, np.__file__)
_ = np.random.RandomState(0).rand(1)
if not np.__version__.startswith("1.26."):
    raise RuntimeError(
        f"Expected NumPy 1.26.x in this notebook, got {np.__version__}. "
        "Rerun the dependency cell, then Runtime -> Restart runtime if this persists."
    )

import torch
from ultralytics import YOLO
import boxmot
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from logo_removal.mask_providers import CrowdedHumanMaskProvider
from logo_removal.vlm_analysis import VALID_VLM_PROVIDERS

boxmot_version = metadata.version("boxmot")
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Quality restoration mode:", QUALITY_RESTORATION)
required_quality_template = {
    "realesrgan": "REALESRGAN_COMMAND_TEMPLATE",
    "venhancer": "VENHANCER_COMMAND_TEMPLATE",
}.get(QUALITY_RESTORATION)
if required_quality_template and not os.environ.get(required_quality_template):
    raise RuntimeError(
        f"QUALITY_RESTORATION={QUALITY_RESTORATION!r} requires matching quality restoration template "
        f"{required_quality_template}. Run the quality restoration setup cell first."
    )
print("Quality restoration template:", required_quality_template or "not required")
print("VALID_VLM_PROVIDERS:", sorted(VALID_VLM_PROVIDERS))
if VLM_PROVIDER == "ollama":
    wait_for_ollama(timeout=30)
    available_ollama_models = get_ollama_model_names()
    selected_ollama_model = os.environ.get("OLLAMA_VLM_MODEL", OLLAMA_VLM_MODEL)
    print("Ollama VLM model:", selected_ollama_model)
    print("Available Ollama models:", sorted(available_ollama_models))
    if selected_ollama_model not in available_ollama_models:
        raise RuntimeError(
            f"Ollama model {selected_ollama_model!r} is not downloaded. Run the Ollama model download cell before preflight."
        )
print("BoxMOT version:", boxmot_version)
print("BoxMOT package:", getattr(boxmot, "__file__", "<unknown>"))
tracker_zoo = importlib.import_module("boxmot.trackers.tracker_zoo")
print("BoxMOT tracker_zoo:", getattr(tracker_zoo, "__file__", "<unknown>"))

if boxmot_version.startswith("20."):
    raise RuntimeError(
        "BoxMOT 20.x is installed, but this Colab notebook pins NumPy 1.26.4. "
        "BoxMOT 20 requires NumPy >=2.2.0. Rerun the dependency cell, confirm it installs boxmot>=19,<20, "
        "then Runtime -> Restart runtime and rerun from the top."
    )

crowded_human_sam2_size = int(os.environ.get("CROWDED_HUMAN_SAM2_IMAGE_SIZE", "1024"))
print("Crowded-human SAM2 image size:", crowded_human_sam2_size)
if crowded_human_sam2_size != 1024:
    raise RuntimeError(
        "Crowded-human SAM2 image refinement must use CROWDED_HUMAN_SAM2_IMAGE_SIZE=1024; "
        "512 causes image/prompt embedding grid mismatch in SAM2ImagePredictor."
    )

print("Warming YOLO detector:", CROWDED_HUMAN_DETECTOR_MODEL)
_ = YOLO(CROWDED_HUMAN_DETECTOR_MODEL)

reid_ref = os.environ.get("CROWDED_HUMAN_REID_MODEL", CROWDED_HUMAN_REID_MODEL)
reid_path = Path(reid_ref)
if not reid_path.is_absolute():
    candidates = [
        PROJECT_ROOT / "models" / "reid" / reid_ref,
        PROJECT_ROOT / reid_ref,
        Path("/content") / reid_ref,
    ]
    reid_path = next((candidate for candidate in candidates if candidate.exists()), Path(reid_ref))

if reid_path.exists():
    print(f"Using ReID weights at: {reid_path} ({reid_path.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    print(f"Using ReID model name; BoxMOT may download it: {reid_ref}")
    reid_path = Path(reid_ref)

# Warm the exact BoxMOT path used by the app. This supports BoxMOT v19 and guarded fallbacks.
provider = object.__new__(CrowdedHumanMaskProvider)
factory_count = len(provider._resolve_boxmot_create_tracker_functions(boxmot))
class_count = len(provider._resolve_boxmot_botsort_classes(boxmot))
print(f"Resolved BoxMOT factories: {factory_count}; BoT-SORT classes: {class_count}")
tracker = provider._build_botsort_tracker(
    boxmot_module=boxmot,
    reid_weights=str(reid_path),
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("BoxMOT BoT-SORT initialized:", type(tracker))
del tracker
print("Preflight OK")


## 12. Start Celery Worker

The web API queues every video job through Celery, so start Redis and one solo worker before launching FastAPI.


In [ ]:
run(["service", "redis-server", "start"])

if "CELERY_WORKER_PROCESS" in globals() and CELERY_WORKER_PROCESS.poll() is None:
    print("Stopping existing Celery worker...")
    CELERY_WORKER_PROCESS.terminate()
    try:
        CELERY_WORKER_PROCESS.wait(timeout=10)
    except subprocess.TimeoutExpired:
        CELERY_WORKER_PROCESS.kill()
        CELERY_WORKER_PROCESS.wait(timeout=5)

worker_env = os.environ.copy()
worker_env["PYTHONUNBUFFERED"] = "1"
worker_env["PYTHONPATH"] = f"{PROJECT_ROOT / 'src'}:{SAM2_REPO}:{worker_env.get('PYTHONPATH', '')}"

celery_log_path = PROJECT_ROOT / "celery.log"
celery_log = open(celery_log_path, "a")
CELERY_WORKER_PROCESS = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "celery",
        "-A",
        "src.logo_removal.tasks.celery_app",
        "worker",
        "--loglevel=INFO",
        "--pool=solo",
    ],
    cwd=PROJECT_ROOT,
    env=worker_env,
    stdout=celery_log,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print("Celery worker started.")
print("Celery log:", celery_log_path)

def stop_celery_worker():
    process = globals().get("CELERY_WORKER_PROCESS")
    if process is None or process.poll() is not None:
        print("Celery worker is not running.")
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)
    print("Celery worker stopped.")

atexit.register(stop_celery_worker)


## 13. Start FastAPI UI

Open the printed Colab proxy URL. In the UI choose:

- Pipeline: `SAM2 + VOID`
- Removal mode: `Crowded Human` for the Phase 2 path
- Affected regions: `Ollama VLM`
- Run VOID pass: checked, when you want Colab to run the full VOID model


In [ ]:
HOST = "127.0.0.1"
PORT = 8000
SERVER_URL = f"http://{HOST}:{PORT}"


def _stream_process_logs(process):
    assert process.stdout is not None
    for line in process.stdout:
        print(line.rstrip())


def _healthcheck(url, timeout=40):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f"{url}/api/health", timeout=1) as response:
                if response.status == 200:
                    return True
        except Exception as exc:
            last_error = exc
            time.sleep(0.5)
    raise RuntimeError(f"FastAPI did not become ready at {url}: {last_error}")


if "MAIN_SERVER_PROCESS" in globals() and MAIN_SERVER_PROCESS.poll() is None:
    print(f"FastAPI is already running at {SERVER_URL}")
else:
    server_env = os.environ.copy()
    server_env["PYTHONUNBUFFERED"] = "1"
    server_env["PYTHONPATH"] = f"{PROJECT_ROOT / 'src'}:{SAM2_REPO}:{server_env.get('PYTHONPATH', '')}"
    MAIN_SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "main.py"],
        cwd=PROJECT_ROOT,
        env=server_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    MAIN_SERVER_LOG_THREAD = threading.Thread(
        target=_stream_process_logs,
        args=(MAIN_SERVER_PROCESS,),
        daemon=True,
    )
    MAIN_SERVER_LOG_THREAD.start()
    _healthcheck(SERVER_URL)
    print(f"FastAPI UI is ready locally: {SERVER_URL}")

try:
    from google.colab import output
    proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print("Open this Colab proxy URL:", proxy_url)
except Exception as exc:
    print("Could not create Colab proxy URL:", exc)


def stop_main_server():
    process = globals().get("MAIN_SERVER_PROCESS")
    if process is None or process.poll() is not None:
        print("FastAPI server is not running.")
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)
    print("FastAPI server stopped.")

atexit.register(stop_main_server)


## 14. Optional Public ngrok Tunnel

Use this only if the Colab proxy URL is not enough. Do not hardcode tokens into the notebook; paste your token into `NGROK_AUTHTOKEN` at runtime.


In [ ]:
NGROK_AUTHTOKEN = ""  # Paste your ngrok token here only for the current runtime if needed.
START_NGROK = False

if START_NGROK:
    from pyngrok import ngrok
    ngrok.kill()
    if NGROK_AUTHTOKEN:
        ngrok.set_auth_token(NGROK_AUTHTOKEN)
    tunnel = ngrok.connect(PORT)
    print("Public FastAPI URL:", tunnel.public_url)
else:
    print("ngrok tunnel skipped. Use the Colab proxy URL from the previous cell.")


## 15. Optional Direct CLI Run

The UI is usually easier for choosing the ROI. Use this cell when you already know the input path and ROI and want the whole Phase 1-3 pipeline from the command line.


In [ ]:
RUN_DIRECT_PIPELINE = False
DIRECT_INPUT_VIDEO = Path("/content/drive/MyDrive/input_video.mp4")
DIRECT_OUTPUT_DIR = COLAB_OUTPUT_ROOT / "direct_full_void_pipeline"
DIRECT_REMOVAL_MODE = "ai_human_crowded"  # ai_human_crowded | ai_object | ai_text_or_logo | static_rectangle
DIRECT_TRACKING_BACKEND = "samurai"
DIRECT_REFERENCE_FRAME = 5
DIRECT_ROI_X = 278
DIRECT_ROI_Y = 104
DIRECT_ROI_WIDTH = 77
DIRECT_ROI_HEIGHT = 215
DIRECT_MAX_CHUNKS = None  # Set to 1 for a quick smoke test.
DIRECT_RUN_VOID = True

if RUN_DIRECT_PIPELINE:
    if not DIRECT_INPUT_VIDEO.exists():
        raise FileNotFoundError(f"Input video not found: {DIRECT_INPUT_VIDEO}")
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "run_full_void_pipeline.py"),
        "--input", str(DIRECT_INPUT_VIDEO),
        "--output-dir", str(DIRECT_OUTPUT_DIR),
        "--removal-mode", DIRECT_REMOVAL_MODE,
        "--sam2-tracking-backend", DIRECT_TRACKING_BACKEND,
        "--reference-frame", str(DIRECT_REFERENCE_FRAME),
        "--mask-padding", "2",
        "--resource-profile", RESOURCE_PROFILE,
        "--quality-restoration", QUALITY_RESTORATION,
        "--vlm-provider", VLM_PROVIDER,
        "--heuristic-contact-dilation-px", "10",
        "--heuristic-shadow-dilation-px", "30",
        "--heuristic-shadow-vertical-offset-px", "16",
        "--x", str(DIRECT_ROI_X),
        "--y", str(DIRECT_ROI_Y),
        "--width", str(DIRECT_ROI_WIDTH),
        "--height", str(DIRECT_ROI_HEIGHT),
        "--void-repo", str(VOID_REPO),
        "--colab-data-root", "/content/void_phase5_data",
        "--colab-output-dir", "/content/void_phase5_outputs",
        "--colab-upload-dir", "/content/void_phase5_upload",
        "--colab-chunk-outputs-dir", "/content/void_phase5_chunk_outputs",
        "--colab-merged-output-path", "/content/void_phase5_merged.mp4",
    ]
    if DIRECT_MAX_CHUNKS is not None:
        cmd.extend(["--max-chunks", str(DIRECT_MAX_CHUNKS)])
    if DIRECT_RUN_VOID:
        cmd.append("--run-void")
    run_stream(cmd, cwd=PROJECT_ROOT, env=os.environ.copy())
else:
    print("Direct CLI run skipped. Set RUN_DIRECT_PIPELINE = True after filling input path and ROI.")


## 16. Health, Logs, And Stop Cells


In [ ]:
with urllib.request.urlopen(f"{SERVER_URL}/api/health", timeout=3) as response:
    print(response.status, response.read().decode("utf-8"))


In [ ]:
!tail -n 200 -f celery.log


In [ ]:
stop_main_server()
stop_celery_worker()
